# Análise de Veículos Elétricos (EV Analytics)

Este notebook realiza uma análise completa da base `electric_vehicle_analytics.csv`, dividida em três partes:

1. **Exploração estatística e comparação de grupos**
2. **Tendências de mercado ao longo do tempo**
3. **Modelagem preditiva do valor de revenda**

O objetivo é entender o perfil dos veículos elétricos, observar a evolução de indicadores de negócio ao longo dos anos e construir um modelo capaz de estimar o `Resale_Value_USD` (valor de revenda) a partir das características do veículo.


## 0. Configuração inicial

Importação das bibliotecas utilizadas ao longo do notebook e configuração de estilo dos gráficos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42


## 1. Carregamento e preparação dos dados

O dataset contém informações de veículos elétricos, incluindo características técnicas (bateria, autonomia, velocidade), 
custos (manutenção, seguro, carregamento), impacto ambiental (CO₂ economizado) e o valor de revenda.


In [ ]:
df = pd.read_csv("data/electric_vehicle_analytics.csv")
print(f"Dimensão do dataset: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head()


In [ ]:
# Tipos de dados e valores nulos
info_df = pd.DataFrame({
    "dtype": df.dtypes,
    "n_nulos": df.isnull().sum(),
    "% nulos": (df.isnull().mean() * 100).round(2)
})
info_df


**Observação:** o dataset não apresenta valores nulos em nenhuma coluna, portanto não é necessária nenhuma estratégia de imputação. Vamos apenas remover a coluna `Vehicle_ID`, que é um identificador sem valor analítico, das análises estatísticas (ela será mantida no DataFrame original, mas descartada nos modelos).

In [ ]:
# Estatísticas descritivas das variáveis numéricas
df.describe().T


In [ ]:
# Estatísticas descritivas das variáveis categóricas
df.describe(include="object").T


In [ ]:
# Separando colunas numéricas e categóricas (Vehicle_ID e Year tratados à parte)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols.remove("Vehicle_ID")
cat_cols = df.select_dtypes(include="object").columns.tolist()

print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)


---
# Parte 1 — Exploração estatística e comparação de grupos

Nesta seção vamos:
- Estudar a distribuição das variáveis numéricas e categóricas;
- Comparar grupos (ex.: tipo de veículo, região, tipo de uso) por meio de testes estatísticos;
- Investigar associações entre variáveis categóricas e correlações entre variáveis numéricas.


### 1.1 Distribuição das variáveis categóricas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
cat_plot_cols = ["Make", "Region", "Vehicle_Type", "Usage_Type"]

for ax, col in zip(axes.flatten(), cat_plot_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, ax=ax, hue=col, legend=False, palette="viridis")
    ax.set_title(f"Distribuição por {col}")
    ax.set_xlabel("Quantidade de veículos")

plt.tight_layout()
plt.show()


**Insight:** a base está razoavelmente distribuída entre as marcas, regiões, tipos de veículo e tipos de uso, sem uma categoria dominante extrema — o que favorece comparações estatísticas justas entre os grupos.

### 1.2 Distribuição das variáveis numéricas

In [ ]:
num_plot_cols = ["Battery_Capacity_kWh", "Range_km", "Battery_Health_%",
                  "Maintenance_Cost_USD", "Insurance_Cost_USD", "Resale_Value_USD"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flatten(), num_plot_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="teal")
    ax.set_title(f"Distribuição de {col}")

plt.tight_layout()
plt.show()


**Insight:** as variáveis numéricas principais (capacidade de bateria, autonomia, custos e valor de revenda) apresentam distribuições aproximadamente contínuas e sem outliers extremos aparentes, sugerindo dados sintéticos bem comportados, adequados para testes paramétricos e modelagem.

### 1.3 Comparação de grupos: Valor de revenda por características do veículo

Vamos comparar a distribuição de `Resale_Value_USD` entre diferentes categorias usando boxplots e testes de hipótese (ANOVA).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sns.boxplot(data=df, x="Vehicle_Type", y="Resale_Value_USD", ax=axes[0], hue="Vehicle_Type", legend=False, palette="viridis")
axes[0].set_title("Valor de revenda por Tipo de Veículo")

sns.boxplot(data=df, x="Region", y="Resale_Value_USD", ax=axes[1], hue="Region", legend=False, palette="viridis")
axes[1].set_title("Valor de revenda por Região")
axes[1].tick_params(axis='x', rotation=20)

sns.boxplot(data=df, x="Usage_Type", y="Resale_Value_USD", ax=axes[2], hue="Usage_Type", legend=False, palette="viridis")
axes[2].set_title("Valor de revenda por Tipo de Uso")

plt.tight_layout()
plt.show()


In [ ]:
# Teste ANOVA: Resale_Value_USD entre grupos de Vehicle_Type
grupos_tipo = [g["Resale_Value_USD"].values for _, g in df.groupby("Vehicle_Type")]
f_tipo, p_tipo = stats.f_oneway(*grupos_tipo)

# Teste ANOVA: Resale_Value_USD entre grupos de Region
grupos_regiao = [g["Resale_Value_USD"].values for _, g in df.groupby("Region")]
f_regiao, p_regiao = stats.f_oneway(*grupos_regiao)

# Teste ANOVA: Resale_Value_USD entre grupos de Usage_Type
grupos_uso = [g["Resale_Value_USD"].values for _, g in df.groupby("Usage_Type")]
f_uso, p_uso = stats.f_oneway(*grupos_uso)

resultados_anova = pd.DataFrame({
    "Comparação": ["Resale_Value ~ Vehicle_Type", "Resale_Value ~ Region", "Resale_Value ~ Usage_Type"],
    "Estatística F": [f_tipo, f_regiao, f_uso],
    "p-valor": [p_tipo, p_regiao, p_uso]
})
resultados_anova["Significativo (alfa=0.05)"] = resultados_anova["p-valor"] < 0.05
resultados_anova


**Interpretação dos testes ANOVA:**
- **Tipo de veículo:** o p-valor obtido é bem superior a 0,05, portanto **não há evidência estatística** de que o tipo de veículo (SUV, Sedan, Hatchback, Truck) influencie o valor médio de revenda.
- **Região:** o p-valor fica abaixo de 0,05, indicando uma **diferença estatisticamente significativa**, ainda que pequena, entre as médias de revenda por região — algo a ser observado com cautela, pois o tamanho do efeito é discreto.
- **Tipo de uso:** o p-valor é superior a 0,05, ou seja, **não há diferença significativa** no valor de revenda entre veículos de uso pessoal, comercial ou de frota.

Esses resultados sugerem que o valor de revenda é definido, principalmente, por características técnicas do veículo (como veremos a seguir), e não pela categoria de uso ou tipo de carroceria.

### 1.4 Associação entre variáveis categóricas (Qui-quadrado)

Vamos verificar se existe associação entre `Vehicle_Type` e `Region`, ou seja, se certos tipos de veículos são mais comuns em certas regiões.

In [ ]:
tabela_contingencia = pd.crosstab(df["Vehicle_Type"], df["Region"])
chi2, p_chi2, dof, esperado = stats.chi2_contingency(tabela_contingencia)

print(f"Estatística Qui-quadrado: {chi2:.2f}")
print(f"Graus de liberdade: {dof}")
print(f"p-valor: {p_chi2:.4f}")

plt.figure(figsize=(8, 5))
sns.heatmap(tabela_contingencia, annot=True, fmt="d", cmap="viridis")
plt.title("Tabela de contingência: Tipo de Veículo x Região")
plt.show()


**Interpretação:** o p-valor do teste Qui-quadrado é superior a 0,05, portanto **não rejeitamos a hipótese nula** de independência: não há associação estatisticamente significativa entre o tipo de veículo e a região — a distribuição de tipos de veículo é semelhante em todas as regiões.

### 1.5 Correlação entre variáveis numéricas

Por fim, vamos observar como as variáveis numéricas se correlacionam entre si, com foco especial na variável-alvo `Resale_Value_USD`, que será utilizada na Parte 3.

In [ ]:
corr = df[num_cols].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Matriz de correlação entre variáveis numéricas")
plt.show()

corr["Resale_Value_USD"].sort_values(ascending=False)


**Insight central da Parte 1:** o `Resale_Value_USD` apresenta correlação forte e positiva com `Battery_Capacity_kWh` (capacidade da bateria) e `Range_km` (autonomia), e correlação moderada com o `Year` (ano do veículo — carros mais novos valem mais). As demais variáveis (custos, velocidade, número de ciclos de carga, saúde da bateria) apresentam correlação muito fraca com o valor de revenda. Isso já indica quais variáveis provavelmente dominarão o modelo preditivo na Parte 3.

---
# Tendências de Mercado

Nesta seção observamos como os principais indicadores de negócio evoluíram entre 2015 e 2024, e projetamos os próximos anos com uma regressão linear simples sobre o tempo.


### 2.1 Agregações anuais dos principais indicadores

In [ ]:
indicadores_anuais = df.groupby("Year").agg(
    Qtd_Veiculos=("Vehicle_ID", "count"),
    Resale_Medio=("Resale_Value_USD", "mean"),
    CO2_Medio_ton=("CO2_Saved_tons", "mean"),
    Manutencao_Media=("Maintenance_Cost_USD", "mean"),
    Seguro_Medio=("Insurance_Cost_USD", "mean"),
    Custo_Carga_Mensal_Medio=("Monthly_Charging_Cost_USD", "mean"),
    Autonomia_Media_km=("Range_km", "mean"),
    Saude_Bateria_Media=("Battery_Health_%", "mean"),
).reset_index().round(2)

indicadores_anuais


### 2.2 Evolução dos indicadores ao longo do tempo

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.lineplot(data=indicadores_anuais, x="Year", y="Resale_Medio", marker="o", ax=axes[0,0], color="teal")
axes[0,0].set_title("Valor médio de revenda por ano (USD)")

sns.lineplot(data=indicadores_anuais, x="Year", y="CO2_Medio_ton", marker="o", ax=axes[0,1], color="darkgreen")
axes[0,1].set_title("CO2 médio economizado por ano (toneladas)")

sns.lineplot(data=indicadores_anuais, x="Year", y="Manutencao_Media", marker="o", ax=axes[1,0], color="orange", label="Manutenção")
sns.lineplot(data=indicadores_anuais, x="Year", y="Seguro_Medio", marker="o", ax=axes[1,0], color="crimson", label="Seguro")
axes[1,0].set_title("Custos médios anuais (USD)")
axes[1,0].set_ylabel("USD")
axes[1,0].legend()

sns.lineplot(data=indicadores_anuais, x="Year", y="Autonomia_Media_km", marker="o", ax=axes[1,1], color="purple")
axes[1,1].set_title("Autonomia média por ano (km)")

plt.tight_layout()
plt.show()


**Leitura dos gráficos:**
- O **valor médio de revenda** apresenta tendência de crescimento praticamente contínua ao longo dos anos, refletindo a evolução tecnológica dos veículos elétricos (baterias maiores, mais autonomia).
- O **CO₂ economizado** oscila em torno de uma média estável, sem tendência forte de alta ou queda — indica que o benefício ambiental por veículo já está relativamente estabilizado na base.
- Os **custos médios de manutenção e seguro** não mostram uma tendência clara de aumento ou queda, oscilando ano a ano.
- A **autonomia média** também se mantém relativamente estável, com pequenas oscilações, sem uma tendência de crescimento tão acentuada quanto o valor de revenda — sugerindo que o aumento no valor de revenda é influenciado por outros fatores além da autonomia isolada (como a capacidade de bateria e o próprio ano/geração do veículo).


### 2.3 Projeção simples com regressão linear

Usamos uma regressão linear simples (`Indicador ~ Ano`) para projetar o comportamento do valor médio de revenda e do CO₂ médio economizado nos próximos 3 anos (2025–2027). Trata-se de uma projeção ilustrativa, útil para visualizar a tendência, não uma previsão robusta.

In [ ]:
from sklearn.linear_model import LinearRegression

anos_futuros = np.array([2025, 2026, 2027]).reshape(-1, 1)

def projetar_regressao_linear(df_anual, coluna, anos_futuros):
    X = df_anual["Year"].values.reshape(-1, 1)
    y = df_anual[coluna].values
    modelo = LinearRegression().fit(X, y)
    y_pred_hist = modelo.predict(X)
    y_pred_futuro = modelo.predict(anos_futuros)
    r2 = modelo.score(X, y)
    return modelo, y_pred_hist, y_pred_futuro, r2

modelo_resale, pred_hist_resale, pred_futuro_resale, r2_resale = projetar_regressao_linear(
    indicadores_anuais, "Resale_Medio", anos_futuros
)
modelo_co2, pred_hist_co2, pred_futuro_co2, r2_co2 = projetar_regressao_linear(
    indicadores_anuais, "CO2_Medio_ton", anos_futuros
)

print(f"Regressão Resale_Medio ~ Year | R² = {r2_resale:.3f} | coeficiente anual = {modelo_resale.coef_[0]:.2f} USD/ano")
print(f"Projeção 2025-2027 (Resale_Medio): {np.round(pred_futuro_resale, 2)}")
print()
print(f"Regressão CO2_Medio ~ Year | R² = {r2_co2:.3f} | coeficiente anual = {modelo_co2.coef_[0]:.4f} ton/ano")
print(f"Projeção 2025-2027 (CO2_Medio_ton): {np.round(pred_futuro_co2, 2)}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

anos_completos = np.concatenate([indicadores_anuais["Year"].values, anos_futuros.flatten()])

axes[0].scatter(indicadores_anuais["Year"], indicadores_anuais["Resale_Medio"], color="teal", label="Observado")
axes[0].plot(indicadores_anuais["Year"], pred_hist_resale, color="black", linestyle="--", label="Ajuste linear")
axes[0].plot(anos_futuros.flatten(), pred_futuro_resale, color="red", marker="o", linestyle="--", label="Projeção 2025-2027")
axes[0].set_title("Projeção do valor médio de revenda")
axes[0].set_xlabel("Ano")
axes[0].set_ylabel("USD")
axes[0].legend()

axes[1].scatter(indicadores_anuais["Year"], indicadores_anuais["CO2_Medio_ton"], color="darkgreen", label="Observado")
axes[1].plot(indicadores_anuais["Year"], pred_hist_co2, color="black", linestyle="--", label="Ajuste linear")
axes[1].plot(anos_futuros.flatten(), pred_futuro_co2, color="red", marker="o", linestyle="--", label="Projeção 2025-2027")
axes[1].set_title("Projeção do CO2 médio economizado")
axes[1].set_xlabel("Ano")
axes[1].set_ylabel("Toneladas")
axes[1].legend()

plt.tight_layout()
plt.show()


### Principais achados — Tendências de Mercado

1. **Valorização crescente:** o valor médio de revenda dos veículos elétricos cresceu de forma consistente entre 2015 e 2024, com um ajuste linear que explica boa parte dessa tendência (R² elevado). A projeção para 2025–2027 indica continuidade desse crescimento, caso o padrão histórico se mantenha.
2. **Impacto ambiental estável:** o CO₂ médio economizado por veículo não mostra tendência clara de crescimento — o ajuste linear tem baixo poder explicativo (R² baixo), o que sugere que esse indicador está mais associado a características individuais do veículo (uso, quilometragem) do que a uma evolução ao longo dos anos.
3. **Custos operacionais estáveis:** manutenção e seguro não apresentam tendência de alta ao longo do tempo, o que é positivo do ponto de vista de custo total de propriedade (TCO) para o consumidor.
4. **Recomendação de negócio:** como o valor de revenda tende a crescer e está fortemente ligado à capacidade de bateria (ver Parte 1), investir em veículos com baterias maiores tende a preservar melhor o valor do ativo ao longo do tempo — um argumento útil tanto para fabricantes quanto para frotas/locadoras que revendem veículos.

> ⚠️ **Cautela:** a projeção linear é uma simplificação. Ela assume que a tendência histórica se mantém constante, o que pode não capturar mudanças bruscas de mercado (ex.: novas tecnologias de bateria, políticas públicas, oscilações macroeconômicas).


---
# Parte 3 — Modelagem Preditiva

O objetivo desta seção é construir um modelo de machine learning capaz de prever o **valor de revenda (`Resale_Value_USD`)** de um veículo elétrico a partir de suas características técnicas, de uso e custos.

Etapas:
1. Preparação das variáveis (encoding de categóricas, padronização de numéricas);
2. Divisão treino/teste;
3. Treinamento de diferentes algoritmos de regressão;
4. Combinação de modelos (ensemble) e comparação de métricas;
5. Análise de importância das variáveis;
6. Recomendações de negócio.


### 3.1 Preparação dos dados

Removemos `Vehicle_ID` (identificador sem valor preditivo) e separamos a variável-alvo `Resale_Value_USD`. As variáveis categóricas (`Make`, `Model`, `Region`, `Vehicle_Type`, `Usage_Type`) serão tratadas com **One-Hot Encoding**, e as variáveis numéricas serão padronizadas com `StandardScaler` — ambas as transformações são encapsuladas em um `ColumnTransformer` dentro de um `Pipeline`, evitando vazamento de dados (data leakage) entre treino e teste.

In [ ]:
TARGET = "Resale_Value_USD"

X = df.drop(columns=["Vehicle_ID", TARGET])
y = df[TARGET]

cat_features = X.select_dtypes(include="object").columns.tolist()
num_features = X.select_dtypes(include=[np.number]).columns.tolist()

print("Variáveis categóricas:", cat_features)
print("Variáveis numéricas:", num_features)

preprocessador = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")


### 3.2 Treinamento e comparação de algoritmos de regressão

Treinamos quatro algoritmos com naturezas distintas: dois lineares (Regressão Linear e Ridge) e dois baseados em árvores/ensembles (Random Forest e Gradient Boosting).

In [ ]:
modelos = {
    "Regressão Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

pipelines = {}
resultados = []

for nome, modelo in modelos.items():
    pipe = Pipeline([("preprocessador", preprocessador), ("modelo", modelo)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    r2 = r2_score(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))

    pipelines[nome] = pipe
    resultados.append({"Modelo": nome, "R2": r2, "MAE": mae, "RMSE": rmse})

df_resultados = pd.DataFrame(resultados).sort_values("R2", ascending=False).reset_index(drop=True)
df_resultados


### 3.3 Combinação de modelos (Ensemble)

Além dos modelos individuais, testamos um **VotingRegressor**, que combina as previsões de Random Forest, Gradient Boosting e Regressão Linear (média das previsões), buscando um resultado mais robusto do que qualquer modelo isolado.

In [ ]:
ensemble = VotingRegressor(estimators=[
    ("rf", RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
    ("gb", GradientBoostingRegressor(random_state=RANDOM_STATE)),
    ("lr", LinearRegression()),
])

pipe_ensemble = Pipeline([("preprocessador", preprocessador), ("modelo", ensemble)])
pipe_ensemble.fit(X_train, y_train)
pred_ensemble = pipe_ensemble.predict(X_test)

r2_ens = r2_score(y_test, pred_ensemble)
mae_ens = mean_absolute_error(y_test, pred_ensemble)
rmse_ens = np.sqrt(mean_squared_error(y_test, pred_ensemble))

pipelines["Ensemble (Voting)"] = pipe_ensemble
df_resultados = pd.concat([
    df_resultados,
    pd.DataFrame([{"Modelo": "Ensemble (Voting)", "R2": r2_ens, "MAE": mae_ens, "RMSE": rmse_ens}])
], ignore_index=True).sort_values("R2", ascending=False).reset_index(drop=True)

df_resultados


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=df_resultados, x="R2", y="Modelo", hue="Modelo", legend=False, ax=axes[0], palette="viridis")
axes[0].set_title("Comparação de modelos — R² (quanto maior, melhor)")
axes[0].set_xlim(0, 1)

sns.barplot(data=df_resultados, x="RMSE", y="Modelo", hue="Modelo", legend=False, ax=axes[1], palette="magma")
axes[1].set_title("Comparação de modelos — RMSE em USD (quanto menor, melhor)")

plt.tight_layout()
plt.show()


**Interpretação:** todos os modelos atingem um R² próximo de 0,90, o que indica que o conjunto de variáveis disponível explica muito bem o valor de revenda. Os modelos lineares (Regressão Linear e Ridge) e o Ensemble apresentam desempenho ligeiramente superior aos modelos baseados em árvores neste caso, sugerindo que a relação entre as variáveis mais importantes (como `Battery_Capacity_kWh`) e o valor de revenda é predominantemente **linear**. Ainda assim, as diferenças entre os modelos são pequenas.

### 3.4 Escolha do modelo final e análise de resíduos

In [ ]:
melhor_modelo_nome = df_resultados.iloc[0]["Modelo"]
melhor_pipeline = pipelines[melhor_modelo_nome]
pred_melhor = melhor_pipeline.predict(X_test)

print(f"Melhor modelo: {melhor_modelo_nome}")
print(f"R²: {r2_score(y_test, pred_melhor):.4f}")
print(f"MAE: {mean_absolute_error(y_test, pred_melhor):.2f} USD")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, pred_melhor)):.2f} USD")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, pred_melhor, alpha=0.4, color="teal")
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red", linestyle="--")
axes[0].set_xlabel("Valor real (USD)")
axes[0].set_ylabel("Valor previsto (USD)")
axes[0].set_title(f"Real vs. Previsto — {melhor_modelo_nome}")

residuos = y_test - pred_melhor
sns.histplot(residuos, kde=True, ax=axes[1], color="crimson")
axes[1].set_title("Distribuição dos resíduos")
axes[1].set_xlabel("Resíduo (USD)")

plt.tight_layout()
plt.show()


**Interpretação:** os pontos no gráfico Real vs. Previsto se distribuem próximos à linha diagonal (previsão perfeita), sem viés sistemático aparente, e os resíduos apresentam distribuição aproximadamente simétrica em torno de zero — indícios de um modelo bem ajustado e sem erros sistemáticos grosseiros.

### 3.5 Importância das variáveis

Usamos a importância de variáveis (feature importance) do **Random Forest**, que lida naturalmente com variáveis numéricas e categóricas (após o one-hot encoding), para entender quais fatores mais impactam o valor de revenda previsto.

In [ ]:
rf_pipeline = pipelines["Random Forest"]
ohe = rf_pipeline.named_steps["preprocessador"].named_transformers_["cat"]
nomes_features = num_features + list(ohe.get_feature_names_out(cat_features))

importancias = rf_pipeline.named_steps["modelo"].feature_importances_
serie_importancia = pd.Series(importancias, index=nomes_features).sort_values(ascending=False)

top15 = serie_importancia.head(15)

plt.figure(figsize=(10, 7))
sns.barplot(x=top15.values, y=top15.index, hue=top15.index, legend=False, palette="viridis")
plt.title("Top 15 variáveis mais importantes (Random Forest)")
plt.xlabel("Importância relativa")
plt.tight_layout()
plt.show()

top15


**Insight:** de forma consistente com a análise de correlação da Parte 1, a **capacidade da bateria (`Battery_Capacity_kWh`)** é, disparadamente, a variável mais importante para prever o valor de revenda, seguida pelo **ano do veículo (`Year`)**. As demais variáveis (custos, categorias, velocidade, etc.) têm contribuição individual pequena. Isso indica que o valor de revenda de um veículo elétrico é definido, em grande parte, pela sua "geração tecnológica" de bateria — quanto maior a capacidade, maior o valor residual esperado.

---
## Recomendações para o Negócio

Com base em toda a análise (exploração estatística, tendências de mercado e modelagem preditiva), destacamos as seguintes recomendações:

1. **Priorizar capacidade de bateria na composição de frota/portfólio:** como a capacidade de bateria é o principal determinante do valor de revenda (correlação ~0,92 e maior importância no modelo), empresas de locação, frotas corporativas e revendedores devem priorizar veículos com baterias de maior capacidade para preservar valor residual no médio/longo prazo.

2. **Não usar tipo de veículo ou tipo de uso como critério de precificação de revenda:** os testes ANOVA mostraram que `Vehicle_Type` (SUV, Sedan, Hatchback, Truck) e `Usage_Type` (Pessoal, Frota, Comercial) não têm efeito estatisticamente significativo sobre o valor de revenda. Políticas de precificação baseadas nessas categorias podem estar desalinhadas com o real comportamento do mercado.

3. **Monitorar diferenças regionais no valor de revenda:** houve uma diferença estatisticamente significativa (ainda que modesta) entre regiões. Vale investigar se isso reflete diferenças de câmbio, tributação, incentivos governamentais ou preferências locais, e ajustar estratégias comerciais regionais.

4. **Aproveitar a tendência de valorização para planejamento de portfólio:** a série histórica mostra crescimento consistente no valor médio de revenda ano a ano. Empresas de leasing e revenda podem usar essa tendência para dimensionar melhor o momento ideal de renovação de frota, equilibrando depreciação esperada e demanda de mercado.

5. **Usar o modelo preditivo para precificação assistida:** o modelo final (R² ≈ 0,90) pode ser incorporado a uma ferramenta interna de precificação, ajudando equipes comerciais a estimar rapidamente o valor justo de revenda de um veículo com base em suas características técnicas, reduzindo a dependência de avaliações manuais e subjetivas.

6. **Investir em comunicação sobre "saúde de bateria" e autonomia:** embora a saúde da bateria e a autonomia tenham menor peso no modelo do que a capacidade nominal, elas ainda contam entre os fatores técnicos relevantes. Programas de manutenção preventiva e garantias estendidas de bateria podem ajudar a sustentar o valor de revenda ao longo do tempo.
